In [13]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm
import os
import logging
import string

In [3]:
atlanta_rest = pd.read_csv("atlanta_cleaned_and_preproccesed.csv", header=0)

In [23]:
texts = atlanta_rest['clean_text'].astype(str).tolist()

device = 0 
model_name = "Dizex/InstaFoodRoBERTa-NER"

print(f"Loading {model_name} on device {device}...")

ner_pipeline = pipeline(
    "token-classification", 
    model=model_name, 
    aggregation_strategy="first", 
    device=device
)

print(f"Processing {len(texts)} reviews...")

results = []

for output in tqdm(ner_pipeline(texts, batch_size=64), total=len(texts)):
    entities = []
    for r in output:
        if r['score'] > 0.50:
            word = r['word'].lower().strip()
            if len(word) > 2 and "#" not in word:
                entities.append(word)
    
    results.append(list(set(entities)))

atlanta_rest['roberta_food'] = results
atlanta_rest.to_pickle("atlanta_full_roberta_raw.pkl")

Loading Dizex/InstaFoodRoBERTa-NER on device 0...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Processing 50647 reviews...


/usr/local/lib/python3.11/dist-packages/transformers/pipelines/token_classification.py:392: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(
100%|██████████| 50647/50647 [00:00<00:00, 117162.73it/s]
